
# Practice 1 — Unsupervised Learning

**Machine Learning / Ingeniería de Datos I — academic year 2026/2027**

Pair: *(surname 1)*, *(surname 2)*

---

This is a **starter** notebook. It gives you the environment banner, the data paths, the two
helper functions the practical provides, the seal guard, and the section skeleton in the order
you must work in. **Everything marked `TODO` is yours to write**, in code and in Markdown.

Read the statement before you start. Three rules decide a lot of the marks:

1. **Interpretation goes in Markdown cells**, not in Python comments.
2. **The unsupervised seal covers BOTH Zoo exercises.** The order below is not negotiable and
   it is graded: Exercise 1 unsupervised → `FROZEN DECISION — K-MEANS` → Exercise 2
   unsupervised → `FROZEN DECISION — AGGLOMERATIVE` → one common
   `REVEAL — EXTERNAL VALIDATION` stage. **No executable cell above the second freeze may
   load, inspect, reference or derive the Zoo class attribute** — not even to count how many
   classes there are. The skeleton below is already in that order; do not reorder it.
3. **One notebook**, executed **top to bottom in one pass**, with its outputs stored, renamed
   `p1_<surname1>_<surname2>.ipynb`.

Rename this file before you submit it.

## 0. Environment and course-fixed constants

Run this first. It prints the library versions your results were produced with and the three course-fixed seeds. **Do not change `P1_SEEDS`.**

In [1]:
# --- Practice 1: environment banner and course-fixed constants -------------
import sys, os, platform
import numpy as np, matplotlib, sklearn, scipy, pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

P1_SEEDS = [13, 41, 97]          # the three course-fixed seeds -- do not change
SEED = P1_SEEDS[0]               # the single seed, where one is needed

DATA = ".."                      # the Práctica 1 folder, relative to this notebook
ZOO_FEATURES_FILE = os.path.join(DATA, "Files_zoo", "zoo_features.csv")
# the labels path is deliberately NOT defined here -- it is a reveal-stage constant
IMGS  = os.path.join(DATA, "Files_clustering", "images")
FACES = os.path.join(DATA, "Files_PCA", "faces.mat")
TABLE = os.path.join(DATA, "datos_pca_fijo", "breast_cancer.csv")
REDUCED = "reduced_images"

print("python      ", platform.python_version())
print("numpy       ", np.__version__)
print("scipy       ", scipy.__version__)
print("scikit-learn", sklearn.__version__)
print("pandas      ", pd.__version__)
print("matplotlib  ", matplotlib.__version__)
print("Pillow      ", Image.__version__)
print("zoo features", ZOO_FEATURES_FILE)
print("P1_SEEDS    ", P1_SEEDS)

python       3.12.3
numpy        2.5.3
scipy        1.18.1
scikit-learn 1.9.1
pandas       3.0.6
matplotlib   3.11.2
Pillow       12.3.0
zoo features ../Files_zoo/zoo_features.csv
P1_SEEDS     [13, 41, 97]


The two functions below are **provided**. `save_image` is graded in Exercise 4 and must be used **unchanged** — the file-size comparison only means something if every pair writes the same three formats in the same way. Note that its third argument, the palette-colour count, is an **uppercase `K`**: in this course lowercase `k` is the number of features and nothing else.

In [3]:
# --- PROVIDED. Do not modify save_image or display_data. -------------------
# K is the number of palette colours (uppercase, per the course notation:
# lowercase k is the number of FEATURES and nothing else).

def save_image(image, path, K):
	img = Image.fromarray(image)
	img.save(path+".jpg", format="JPEG")
	img.save(path+".png", format="PNG")
	indexed_image = img.convert("P", palette=Image.ADAPTIVE, colors=K)
	indexed_image.save(path+"compressed.png", optimize=True)


def display_data(X):
	example_width = int(np.round(np.sqrt(X.shape[1])))
	example_height = X.shape[1] // example_width

	display_rows = int(np.floor(np.sqrt(X.shape[0])))
	display_cols = int(np.ceil(X.shape[0] / display_rows))

	fig, ax_array = plt.subplots(display_rows, display_cols, figsize=(8, 8))

	for i, ax in enumerate(ax_array.flat):
		if i >= X.shape[0]:
			break
		ax.imshow(X[i].reshape(example_height, example_width), cmap='gray')
		ax.axis('off')
	plt.show()

### The seal guard

This reads a notebook's own JSON and proves that **both** frozen-decision headings exist
exactly once, that the $K$-means freeze comes first, and that the first **executable** access to
the Zoo class attribute comes strictly after the second freeze. Run it at the end, against your
saved file, before you submit.

In [4]:
# --- The unsupervised seal, checked mechanically ---------------------------
# Reads a notebook's own JSON and proves three things:
#   1. both frozen-decision headings exist, exactly once each;
#   2. the K-means freeze precedes the agglomerative freeze;
#   3. the first EXECUTABLE access to the Zoo class labels comes strictly
#      after the agglomerative freeze.
# Only markdown HEADINGS count as a freeze -- prose that merely mentions the
# words does not -- and only CODE cells count as a label access, because the
# seal is a statement about what was executed, not about what was written.
import json, re

FREEZE_KM = "FROZEN DECISION \u2014 K-MEANS"
FREEZE_AG = "FROZEN DECISION \u2014 AGGLOMERATIVE"

# every executable route to the Zoo class attribute
LABEL_PAT = re.compile(
    r"ZOO_TYPE_NAMES|ZOO_LABELS_FILE|load_zoo_labels|zoo_labels|y_true|"
    r"adjusted_rand_score|adjusted_mutual_info_score|"
    r"""\[\s*['"]type['"]\s*\]|\bzoo\.data\b|\bzoo\.names\b""")


def check_seal(nb_path, verbose=True):
    nb = json.load(open(nb_path, encoding="utf-8"))
    cells = nb["cells"]
    src = ["".join(c["source"]) for c in cells]

    def heading(text):
        pat = re.compile(r"^\s*#{1,6}\s*" + re.escape(text) + r"\s*$", re.M)
        return [i for i, s in enumerate(src)
                if cells[i]["cell_type"] == "markdown" and pat.search(s)]

    km, ag = heading(FREEZE_KM), heading(FREEZE_AG)
    if len(km) != 1:
        raise AssertionError(f"expected exactly one `{FREEZE_KM}` heading, found {len(km)}")
    if len(ag) != 1:
        raise AssertionError(f"expected exactly one `{FREEZE_AG}` heading, found {len(ag)}")
    km_at, ag_at = km[0], ag[0]
    if not km_at < ag_at:
        raise AssertionError(
            f"SEAL VIOLATED: the K-means freeze (cell {km_at}) must precede the "
            f"agglomerative freeze (cell {ag_at})")

    uses = [i for i, s in enumerate(src)
            if cells[i]["cell_type"] == "code"
            and LABEL_PAT.search(s)
            and "def check_seal" not in s and "LABEL_PAT" not in s]
    first_use = min(uses) if uses else len(cells)

    if verbose:
        print(f"{FREEZE_KM:34s} -> cell {km_at}")
        print(f"{FREEZE_AG:34s} -> cell {ag_at}")
        print(f"{'first executable label access':34s} -> cell {first_use}")
    if first_use <= ag_at:
        raise AssertionError(
            f"SEAL VIOLATED: code cell {first_use} touches the Zoo class labels "
            f"before the agglomerative freeze at cell {ag_at}")
    if verbose:
        print("SEAL OK: both configurations were frozen before any label was read.")
    return km_at, ag_at, first_use


---
## Exercise 1 — $K$-means on the Zoo database (2.5)

$K$ is the **number of clusters**.

`Files_zoo/zoo_features.csv` is a course file derived from the verified official source. It has
a header, a course-owned integer `row_id`, the display name, and the sixteen predictive
attributes — and **no class column at all**. The field list is in `Files_zoo/zoo_schema.txt`.

`animal_name` is a **display field, not a key**: the official Zoo data contains two rows with
the same name. Align on `row_id` whenever alignment is needed.

**Parts (e) and (f) of this exercise are not here.** They need the class attribute, so they
belong in the reveal stage, after both freezes.

In [ ]:
# `zoo_features.csv` is a course file derived from the verified official source.
# It contains a course-owned integer `row_id`, the display name, and the sixteen
# predictive attributes -- and NO class column at all. There is nothing here to
# drop and nothing to avoid reading: the target is simply not in this file.
#
# `animal_name` is a DISPLAY/NAME field, not a key. The official Zoo data
# contains two rows named `frog`, which its own documentation notes. Use
# `row_id` whenever rows must be aligned.
ZOO_ID = "row_id"
ZOO_NAME = "animal_name"
ZOO_FEATURES = [
    "hair", "feathers", "eggs", "milk", "airborne", "aquatic", "predator",
    "toothed", "backbone", "breathes", "venomous", "fins", "legs", "tail",
    "domestic", "catsize",
]


def load_zoo_features(path=None):
    """The feature-only course file: row_id, animal_name and 16 predictors."""
    frame = pd.read_csv(path or ZOO_FEATURES_FILE)
    assert list(frame.columns) == [ZOO_ID, ZOO_NAME] + ZOO_FEATURES,         f"unexpected columns: {list(frame.columns)}"
    return frame

zoo = load_zoo_features()

n,k = len(zoo) , len(ZOO_FEATURES)

print(f"n = {n} animals (observations)")
print(f"k = {k} predictive attributes")
print(f"Columns in the table: {list(zoo.columns)}")
print(f"Does the target column exist in the file?: {'class' in zoo.columns or 'type' in zoo.columns}")



n = 101 animals (observations)
k = 16 predictive attributes
Columns in the table: ['row_id', 'animal_name', 'hair', 'feathers', 'eggs', 'milk', 'airborne', 'aquatic', 'predator', 'toothed', 'backbone', 'breathes', 'venomous', 'fins', 'legs', 'tail', 'domestic', 'catsize']
Does the target column exist in the file?: False


,row_id,animal_name,hair,feathers,eggs,milk,airborne,aquatic,predator,toothed,backbone,breathes,venomous,fins,legs,tail,domestic,catsize
0,1,aardvark,1,0,0,1,0,0,1,1,1,1,0,0,4,0,0,1
1,2,antelope,1,0,0,1,0,0,0,1,1,1,0,0,4,1,0,1
2,3,bass,0,0,1,0,0,1,1,1,1,0,0,1,0,1,0,0
3,4,bear,1,0,0,1,0,0,1,1,1,1,0,0,4,0,0,1
4,5,boar,1,0,0,1,0,0,1,1,1,1,0,0,4,1,0,1


### 1(a) — loading and preparation

**TODO.** State $n$ and $k$. Say which columns you dropped and why. State your scaling decision, name it (min–max scaling or standardisation), say which columns it applies to, and justify it — `legs` is the one attribute that is not Boolean.

In [10]:
from sklearn.preprocessing import MinMaxScaler

X_zoo = zoo[ZOO_FEATURES].copy()

scaler = MinMaxScaler()

X_zoo[['legs']] = scaler.fit_transform(X_zoo[['legs']])

X_zoo.head()

,hair,feathers,eggs,milk,airborne,aquatic,predator,toothed,backbone,breathes,venomous,fins,legs,tail,domestic,catsize
0,1,0,0,1,0,0,1,1,1,1,0,0,0.5,0,0,1
1,1,0,0,1,0,0,0,1,1,1,0,0,0.5,1,0,1
2,0,0,1,0,0,1,1,1,1,0,0,1,0.0,1,0,0
3,1,0,0,1,0,0,1,1,1,1,0,0,0.5,0,0,1
4,1,0,0,1,0,0,1,1,1,1,0,0,0.5,1,0,1


In [ ]:
# TODO 
from sklearn.cluster import KMeans

K_GRID = [5, 6, 7, 8]

for k in K_GRID:
    inertias = []

    for seed in P1_SEEDS:
        kmeans = KMeans(n_clusters=k, n_init=1, random_state=seed)
        kmeans.fit(X_zoo)
        
        inertias.append(kmeans.inertia_)
    
    mean_inertia = np.mean(inertias)
    spread = np.max(inertias) - np.min(inertias)
    
    print(f"K = {k} | Average inertia: {mean_inertia:.4f} | Spread: {spread:.4f}")

K = 5 | Average inertia: 110.1534 | Spread: 1.2777
K = 6 | Average inertia: 103.2284 | Spread: 8.6616
K = 7 | Average inertia: 91.2443 | Spread: 10.5850
K = 8 | Average inertia: 83.5278 | Spread: 8.7184


### 1(b) — reading the grid

**TODO.** What does the table show? Say explicitly why reporting the *best* of three restarts would be a statement about your luckiest initialisation rather than about $K$.

### Before the freeze: state your two selection rules

Two different questions, and they must not be conflated:

1. **Which $K$?** From the **mean $J$ across all three seeds**, never the minimum. $J$ falls
   with $K$ by construction, so its *level* says nothing; use the **shape** of the curve, the
   cluster sizes, the spread across seeds, or another criterion you can defend.
2. **Which run at that $K$?** Only *after* $K$ is fixed, pick **one** of the three
   course-fixed initialisations to carry forward, by a stated unsupervised rule — for instance
   the lowest $J$ among the three **at the already-frozen $K$**. That is an initialisation
   choice, not a model-order choice, and it must not feed back into question 1.

**TODO — write both rules here, before you apply them.**

In [ ]:
# TODO (c) Compute the UNSUPERVISED evidence for the freeze: cluster sizes across the grid,
#          any curve-shape statistic your rule uses, and the spread across seeds.
#          Then apply rule 1 to fix K, and rule 2 to fix ONE representative seed at that K.
#          Fit that single representative run and keep its labels: the scatter plot, the
#          external validation and the class-as-input comparison must all use it.
K_REF = None
REP_SEED = None
lab_ref = None

In [ ]:
# TODO (c) Scatter plot of two attributes you choose, coloured by the FROZEN representative
#          run's cluster labels.

### 1(c) — reading the plot

**TODO.** What does the plot show, and what does it hide? Be specific about what is lost by looking at two of the $k$ attributes.

## FROZEN DECISION — K-MEANS

**TODO — 1(d). This heading is one of the two the guard checks. Do not rename it.**

Fill this in **before** anything below, and before any cell anywhere reads the class attribute.
The frozen object is **one partition**, not a family of them.

| Field | Value |
|---|---|
| $K$ | *TODO* |
| Representative seed | *TODO* |
| Scaling | *TODO* |
| Rule that chose $K$ | *TODO* |
| Rule that chose the seed | *TODO* |

**Unsupervised argument.** *TODO. "$J$ is smallest at $K = 8$" is not an argument, because $J$
falls as $K$ grows by construction. Say what else you used — and if part of your evidence points
the other way, say that too: a frozen choice with an acknowledged limitation scores better than
one presented as unambiguous.*

In [ ]:
# TODO Freeze the table above in code, so nothing below can drift from it.
FROZEN_KMEANS = dict(K=K_REF, representative_seed=REP_SEED)
print("FROZEN_KMEANS:", FROZEN_KMEANS)


---
## Exercise 2 — Agglomerative clustering on the Zoo database (2.0)

$K$ is again the **number of clusters**. Still unsupervised: **part (d) is not here**, it
belongs in the reveal stage.

In [ ]:
# TODO (a) Agglomerative clustering with every linkage scikit-learn offers.
#          State which metric each linkage admits, and show what happens if you ask for one
#          a linkage does not accept.
from sklearn.cluster import AgglomerativeClustering

LINKAGES = ["ward", "complete", "average", "single"]

### 2(b) — two criteria, both stated before they are applied

This exercise needs **two** unsupervised decisions, not one:

1. **Within each linkage — which $K$?** A different linkage is a different hierarchy, so there
   is no reason for one number to fit all four. Each linkage gets **its own** $K$.
2. **Across linkages — which linkage do you prefer?** State a rule that can actually choose
   one, using only unsupervised evidence. "The one that looks best" is not a rule.

**TODO — write both rules here**, then apply them in the cell below. Do not change either after
the reveal.

In [ ]:
# TODO (b) Apply rule 1 to EACH linkage and record its own K in picks[linkage].
#          Then apply rule 2 to choose ONE preferred linkage.
from scipy.cluster.hierarchy import dendrogram, linkage as scipy_linkage

picks = {}
LK_REF = None          # your preferred linkage, chosen by rule 2

In [ ]:
# TODO (c) One dendrogram, for the PREFERRED linkage, cut at ITS OWN frozen K.

### 2(c) — reading the dendrogram

**TODO.** Which merges are early and cheap, and which one is expensive enough that your rule cut below it?

## FROZEN DECISION — AGGLOMERATIVE

**TODO. The second of the two headings the guard checks. Do not rename it.**

**Two objects are frozen here**, and nothing below may change either.

| Linkage | Frozen $K$ | Why rule 1 landed there |
|---|---|---|
| `ward` | *TODO* | *TODO* |
| `complete` | *TODO* | *TODO* |
| `average` | *TODO* | *TODO* |
| `single` | *TODO* | *TODO* |

**Preferred linkage (rule 2):** *TODO* — and why.

Freezing the preference *before* the labels exist is what makes part (d) a real question: it can
then compare the linkage that **actually** agrees best with the biology against the one you
preferred **without** any biology. They need not be the same, and if they are not, **say so** —
do not go back and edit either rule.

In [ ]:
# TODO Freeze both objects in code.
FROZEN_AGG = dict(picks)          # linkage -> its own frozen K
PREFERRED_LINKAGE = LK_REF        # the frozen cross-linkage preference
print("FROZEN_AGG       :", FROZEN_AGG)
print("PREFERRED_LINKAGE:", PREFERRED_LINKAGE)

In [ ]:
# TODO Record the OBSERVED cluster sizes at each linkage's own frozen K, before the reveal.
#      Part (e) asks what these four linkages actually did on this data, so the evidence for
#      it must exist before any label is read.


---
## REVEAL — EXTERNAL VALIDATION

**This is the first point at which you may touch the Zoo class attribute.** Labels live in
`Files_zoo/zoo_labels.csv`, which is exactly `row_id,type`. Join on **`row_id`** — never on
`animal_name`, which is not unique.

Both configurations above are frozen and nothing below may change them. Recomputing indices for
several $K$, several seeds or several linkages and presenting the best one is an automatic
method penalty.

In [ ]:
# Opened HERE, inside the reveal stage, and nowhere earlier. How many classes
# there are, and what they are called, is itself information the seal withholds.
ZOO_LABELS_FILE = os.path.join(DATA, "Files_zoo", "zoo_labels.csv")


def load_zoo_labels(feature_frame, path=None):
    """`zoo_labels.csv` is exactly row_id,type. Aligned on row_id, never on name."""
    lab = pd.read_csv(path or ZOO_LABELS_FILE)
    assert list(lab.columns) == ["row_id", "type"], f"unexpected: {list(lab.columns)}"
    merged = feature_frame[["row_id"]].merge(lab, on="row_id", how="left", validate="1:1")
    assert merged["type"].notna().all(), "a feature row has no label"
    assert (merged["row_id"].to_numpy() == feature_frame["row_id"].to_numpy()).all(),         "row_id alignment failed"
    return merged["type"].to_numpy()


ZOO_TYPE_NAMES = {1: "mammal", 2: "bird", 3: "reptile", 4: "fish",
                  5: "amphibian", 6: "insect", 7: "invertebrate"}

# TODO Load the class attribute for the first time, with load_zoo_labels(zoo).
from sklearn.metrics import adjusted_rand_score, adjusted_mutual_info_score

y_true = None

In [ ]:
# TODO 1(f) External indices for the FROZEN K-means partition -- the frozen K AND the frozen
#           representative seed -- and for no other. A contingency table is worth building.

### 1(f) — external validation

**TODO.** What would a value near $0$ mean, and one near $1$? Then answer the real question: did the unsupervised evidence lead you to a structure that resembles the biological classes, and **where does it not**?

In [ ]:
# TODO 1(e) Re-fit K-means with the class attribute INCLUDED as an ordinary input feature,
#           at the same frozen K and the same frozen seed, and compare with your frozen run.

### 1(e) — what a label-shaped column does

**TODO.** Do the clusters change? Explain the mechanism: what does a column that encodes the answer do to the distances, and why does its numeric coding matter here?

In [ ]:
# TODO 2(d) External indices for EACH linkage, each fitted at ITS OWN frozen K:
#               AgglomerativeClustering(n_clusters=FROZEN_AGG[lk], linkage=lk)
#           Report the K you used beside each row, and mark which row is PREFERRED_LINKAGE.

### 2(d) — comparison

**TODO.** Two things were fixed before this cell ran. Compare them:

* the linkage with the **best external agreement**, at its own frozen $K$;
* **`PREFERRED_LINKAGE`**, chosen when no label existed.

Do they coincide? **If they do not, that is the answer and it is a good one** — without labels there was no way to know. Do not edit either frozen object to make them agree.

### 2(e) — comparing the four linkages

**TODO.** Using the **observed** cluster sizes and merge behaviour you recorded before the reveal, compare what the four linkages actually did on this data, and explain each from what that linkage measures between two groups.

`single` uses the **closest** pair, so one intermediate observation can bridge two distant groups; the classic consequence is **chaining** — one dominant cluster plus a scatter of singletons. **Check whether your run shows it.** If it does, identify it and explain it from the closest-pair definition. **If it does not, say so** — asserting an effect your own table does not show costs the same as missing one that it does.


---
## Exercise 3 — DBSCAN: verifying Problem Sheet 1, Exercise 4 (1.5)

Twelve two-dimensional points, $\varepsilon = 1.0$, *MinPts* $= 4$, and the convention the
sheet fixed: $d_2(p,q) \le \varepsilon$, and **a point counts itself** in its own
neighbourhood.

In [ ]:
# The twelve points of Problem Sheet 1, Exercise 4.
P = np.array([[1.0, 1.0], [1.4, 1.0], [1.0, 1.4], [1.4, 1.4], [2.2, 1.2], [3.2, 3.0],
              [5.0, 5.0], [5.4, 5.0], [5.2, 5.4], [5.2, 4.7], [6.1, 5.2], [8.0, 1.5]])
EPS, MINPTS = 1.0, 4

# TODO (a) Compute the pairwise distances and, for each point, the size and membership of its
#          eps-neighbourhood UNDER THE SHEET'S CONVENTION. Classify core / border / noise.

In [ ]:
# TODO (b) sklearn.cluster.DBSCAN with the same eps and min_samples.
#          Report labels_ and core_sample_indices_, and derive core / border / noise from them.
from sklearn.cluster import DBSCAN

In [ ]:
# TODO (c) One table: hand answer | your part (a) | the library, for all twelve points.

### 3(d) — diagnosis

**TODO.** If anything disagrees, say **which** of the three causes it is — the neighbourhood convention, another implementation convention, or an error in your own code — with evidence. If everything agrees, **show why it had to**: name the convention the library uses and demonstrate that it is the sheet's.

*Blaming a disagreement on the library without evidence scores 0 here.*

In [ ]:
# TODO (e) Recompute under the OPPOSITE convention: a point does not count itself.

### 3(e) — why a convention changes the answer

**TODO.** One or two sentences.


---
## Exercise 4 — Colour quantisation and file size (2.0)

$K$ is the **number of palette colours**.

In [ ]:
# TODO (a) Implement the three helpers. save_image is already provided above -- do not rewrite it.

def load_image(path):
    """Load a JPG or PNG image and return it as a numpy array."""
    raise NotImplementedError


def show_image(image):
    """Display the image held in a numpy array."""
    raise NotImplementedError


def get_size(path):
    """Return the size of a file in KB."""
    raise NotImplementedError

In [ ]:
GENERAL   = [3, 5, 10, 16, 20, 32, 50, 64]
LANDSCAPE = [5, 10, 20]
IMAGE_GRID = {"Baboon.png": GENERAL, "Lena.png": GENERAL, "Peppers.png": GENERAL,
              "paisaje.jpg": LANDSCAPE, "paisaje2.jpg": LANDSCAPE}
SUBSAMPLE = 20_000     # pixels used to FIT K-means; every pixel is then assigned

# TODO (b) Quantise. Fit K-means on a random subsample of pixels drawn with SEED,
#          assign every pixel to the nearest fitted centroid, and save with save_image
#          into reduced_images/ following the pattern originalname_K.
os.makedirs(REDUCED, exist_ok=True)

In [ ]:
# TODO (c) For each image, plot size in KB against K with one curve per format
#          (jpg, png, indexed png), and mark the size of the original file.

### 4(d) — interpretation

**TODO.** Answer all four questions of the statement:

1. Which format shrinks as $K$ falls and which barely responds at all — explained from how each format encodes an image.
2. Why the JPG at small $K$ can be **larger** than at a much larger $K$.
3. What property of the content makes `Baboon.png` behave differently from `Lena.png`.
4. Looking at the images: at roughly which $K$ does each stop looking obviously damaged, and does that agree with where the size curve flattens?